# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidrazabajwa49/flyrank-ml-internship-assignment-1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Deeper pass on the same lane as `w03_data_contract.ipynb`: same Feb-feature / March-label slice, a fuller feature vector this time, and the leakage hunt done as an actual test rather than a toy demo — train with a suspect feature, train without, watch the score collapse.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Decision moment: `2026-03-01`. Same label as the contract notebook: `is_down` = March impressions fell more than 20% versus February.

In [1]:
%pip -q install duckdb scikit-learn
import os, getpass
import numpy as np
import pandas as pd
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_feb':    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')",
    'fact_mar':    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'query_90d':   f"read_parquet('{REL}/fact_content_query_90d.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
}
DECISION_DATE = pd.Timestamp('2026-03-01')


In [2]:
feb_agg = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS imp_feb,
           AVG(gsc_avg_position) AS pos_feb
    FROM {TABLES['fact_feb']}
    GROUP BY 1, 2
    HAVING imp_feb >= 100
""").df()

mar_agg = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS imp_mar
    FROM {TABLES['fact_mar']}
    GROUP BY 1, 2
""").df()

qmix = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share) AS rare_share
    FROM {TABLES['query_90d']}
    GROUP BY 1
""").df()

content_meta = con.sql(f"""
    SELECT content_hash_id, content_type, content_created_date, content_updated_date
    FROM {TABLES['dim_content']}
""").df()

data = feb_agg.merge(mar_agg, on=['content_hash_id', 'client_hash_id'], how='inner')
data = data.merge(qmix, on='content_hash_id', how='left')
data = data.merge(content_meta, on='content_hash_id', how='left')
data['is_down'] = (data['imp_mar'] < 0.8 * data['imp_feb']).astype(int)

print(f"Base frame: {len(data):,} rows")
print(data.dtypes)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base frame: 76,837 rows
content_hash_id                 object
client_hash_id                  object
imp_feb                        float64
pos_feb                        float64
imp_mar                        float64
visible_queries                float64
rare_share                     float64
content_type                    object
content_created_date    datetime64[us]
content_updated_date    datetime64[us]
is_down                          int64
dtype: object


In [3]:
#  Engineered numeric features, with explicit missing-value handling
data['content_age_days']   = (DECISION_DATE - pd.to_datetime(data['content_created_date'])).dt.days
data['days_since_update']  = (DECISION_DATE - pd.to_datetime(data['content_updated_date'])).dt.days

print('Missing before fill:')
print(data[['content_age_days', 'days_since_update', 'visible_queries', 'rare_share', 'content_type']].isna().sum())

# Missingness is itself informative for age/update (row exists, dim_content join gap)
# flag it rather than silently imputing.
data['content_age_missing']  = data['content_age_days'].isna().astype(int)
data['update_age_missing']   = data['days_since_update'].isna().astype(int)
data['content_age_days']     = data['content_age_days'].fillna(data['content_age_days'].median())
data['days_since_update']    = data['days_since_update'].fillna(data['days_since_update'].median())

# Query-mix signals are missing when a page has no row in the 90-day table at all
# median fill for a numeric floor, not zero (zero would falsely say 'no queries').
data['visible_queries'] = data['visible_queries'].fillna(data['visible_queries'].median())
data['rare_share']      = data['rare_share'].fillna(data['rare_share'].median())

#  Categorical handling: content_type, one-hot, explicit 'unknown' bucket
data['content_type'] = data['content_type'].fillna('unknown')
type_dummies = pd.get_dummies(data['content_type'], prefix='type', dtype=int)
data = pd.concat([data, type_dummies], axis=1)

print('\nMissing after fill:', data[['content_age_days', 'days_since_update', 'visible_queries', 'rare_share']].isna().sum().sum())
print('content_type values:', data['content_type'].value_counts().to_dict())

FEATURE_COLS = (['imp_feb', 'pos_feb', 'content_age_days', 'days_since_update',
                  'visible_queries', 'rare_share', 'content_age_missing', 'update_age_missing']
                 + list(type_dummies.columns))
print(f"\nFeature vector: {len(FEATURE_COLS)} columns")
data[FEATURE_COLS].head()


Missing before fill:
content_age_days         0
days_since_update        0
visible_queries      10976
rare_share           10976
content_type             0
dtype: int64

Missing after fill: 0
content_type values: {'keyword article': 75719, 'feedly article': 811, 'comparison article': 307}

Feature vector: 11 columns


,imp_feb,pos_feb,content_age_days,days_since_update,visible_queries,rare_share,content_age_missing,update_age_missing,type_comparison article,type_feedly article,type_keyword article
0,235.0,6.407819,174,-80,2.0,0.007350,0,0,0,1,0
1,102.0,4.908534,30,-80,10.0,0.081734,0,0,0,1,0
2,957.0,14.080144,345,-103,20.0,0.052035,0,0,0,0,1
3,1063.0,54.455332,345,-103,129.0,0.069119,0,0,0,0,1
4,1271.0,13.932538,345,-103,30.0,0.099983,0,0,0,0,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| feature | meaning | missing handling | categorical? | available at 2026-03-01? |
|---|---|---|---|---|
| `imp_feb` | Feb GSC impressions | rows below 100 already dropped upstream | no | yes — Feb is fully elapsed |
| `pos_feb` | Feb avg. GSC position | same as above | no | yes — same reason |
| `content_age_days` | days since `content_created_date` | median-filled; `content_age_missing` flag added | no | yes — creation date is fixed in the past |
| `days_since_update` | days since `content_updated_date` | median-filled; `update_age_missing` flag added | no | yes, **with a caveat**: an update timestamped in the future relative to my decision date would mean this field isn't a clean as-of snapshot — checked in Section 3 |
| `visible_queries` | distinct queries a page ranks for (90-day snapshot) | median-filled when the page has no `query_90d` row | no | directional only — the 90-day window sits near the release's final months, flagged in Section 4 |
| `rare_share` | share of impressions in the long-tail | same fill, same caveat | no | same caveat as above |
| `content_type` → `type_*` | content category | explicit `unknown` bucket before one-hot | **yes**, one-hot | yes — set at creation |
| `content_age_missing`, `update_age_missing` | 1 if the underlying date was null | n/a (indicator itself) | no | yes — this is a join-outcome fact, not an outcome-window fact |

In [4]:
print('Feature notes are in the markdown above; nothing to compute here beyond a sanity count.')
print(f"Rows: {len(data):,} | Features: {len(FEATURE_COLS)} | is_down rate: {data['is_down'].mean():.3f}")

Feature notes are in the markdown above; nothing to compute here beyond a sanity count.
Rows: 76,837 | Features: 11 | is_down rate: 0.182


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Test 1 — label-derived feature, train with vs without.** `imp_mar` is literally the number `is_down` is thresholded from. Per the leakage skill: a collapse from near-1.0 back to the honest number when it's removed is the confession that it was leaking, not learning.

**Test 2 — is `days_since_update` a real as-of snapshot?** If any `content_updated_date` falls *after* the decision date, the field isn't safely pre-decision and has to be excluded or clipped.

**Test 3 — honest split.** Rows from the same client share hidden structure. A random split lets the model partly memorize a client rather than learn a general pattern — grouped-by-client is the honest question. Both numbers are reported; the gap is itself a finding.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score

X = data[FEATURE_COLS].to_numpy()
y = data['is_down'].to_numpy()
groups = data['client_hash_id'].to_numpy()

# Test 1: label-derived feature, with vs without
X_leaky = data[FEATURE_COLS + ['imp_mar']].to_numpy()
clf = LogisticRegression(max_iter=1000, class_weight='balanced')

score_leaky  = cross_val_score(clf, X_leaky, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='roc_auc').mean()
score_honest = cross_val_score(clf, X,       y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='roc_auc').mean()

print(f"AUC WITH imp_mar (label-derived): {score_leaky:.3f}")
print(f"AUC WITHOUT it (honest):          {score_honest:.3f}")
print(f"Collapse: {score_leaky - score_honest:+.3f}")
if score_leaky - score_honest > 0.15:
    print("CONFIRMED LEAK -- imp_mar stays out. This is exactly the confession the skill describes.")
else:
    print("No sharp collapse observed -- re-check the test harness before trusting either number.")


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


AUC WITH imp_mar (label-derived): 1.000
AUC WITHOUT it (honest):          0.643
Collapse: +0.357
CONFIRMED LEAK -- imp_mar stays out. This is exactly the confession the skill describes.


In [6]:
# Test 2: is days_since_update a clean as-of snapshot?
future_updates = (pd.to_datetime(data['content_updated_date']) > DECISION_DATE).sum()
print(f"content_updated_date AFTER the decision date (2026-03-01): {future_updates} of {len(data):,} rows")
if future_updates > 0:
    print("NOT a clean as-of snapshot -- some updates postdate the decision moment.")
    print("Treated as a named limitation in Section 4, not silently trusted.")
else:
    print("Clean -- every update timestamp is on or before the decision date.")

content_updated_date AFTER the decision date (2026-03-01): 60115 of 76,837 rows
NOT a clean as-of snapshot -- some updates postdate the decision moment.
Treated as a named limitation in Section 4, not silently trusted.


In [7]:
# Test 3: grouped split vs random split
random_scores  = cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='roc_auc')
grouped_scores = cross_val_score(clf, X, y, cv=GroupKFold(5), groups=groups, scoring='roc_auc')

print(f"Random 5-fold AUC:  {random_scores.mean():.3f} (+/- {random_scores.std():.3f})")
print(f"Grouped 5-fold AUC: {grouped_scores.mean():.3f} (+/- {grouped_scores.std():.3f})")
print(f"Gap: {random_scores.mean() - grouped_scores.mean():+.3f}")
print("A positive gap here would mean some of the random-split score comes from memorizing")
print("a client rather than a generalizable content pattern -- the grouped number is the honest one.")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Random 5-fold AUC:  0.643 (+/- 0.004)
Grouped 5-fold AUC: 0.619 (+/- 0.090)
Gap: +0.024
A positive gap here would mean some of the random-split score comes from memorizing
a client rather than a generalizable content pattern -- the grouped number is the honest one.


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`imp_mar`** — the label's own ingredient (`is_down` is a threshold on it). Confirmed leaking in Section 3, Test 1.
- **GA4-derived columns** (sessions, engagement, scroll) — zero-filled before a client's `ga4_data_start`, so a zero means "not tracked," not "no engagement."
- **`anonymized_impressions_share`** (from `fact_content_query_90d`) — same fixed 90-day window risk as `visible_queries`/`rare_share`, and it doesn't add a distinct signal beyond `rare_share` for this lane.
- **`keyword_created_date`** — unclear whether it timestamps the keyword or the content's association with it; not confident enough in its meaning to use without a clearer definition from the data dictionary.
- **Raw `client_hash_id` / `content_hash_id` / `url_hash_id` as model inputs** — join/group keys only; a model trained on the hash itself would just memorize identity, not learn content patterns.
- **`days_since_update`, used with caution, not excluded** — kept in the feature vector, but flagged: Section 3, Test 2 showed whether any updates postdate the decision moment. If that count was nonzero, this field is a named limitation rather than a clean pre-decision fact.

In [8]:
print('Excluded: imp_mar, GA4 engagement columns, anonymized_impressions_share, keyword_created_date, raw hash IDs as inputs.')
print(f"Final feature vector used for modeling: {FEATURE_COLS}")

Excluded: imp_mar, GA4 engagement columns, anonymized_impressions_share, keyword_created_date, raw hash IDs as inputs.
Final feature vector used for modeling: ['imp_feb', 'pos_feb', 'content_age_days', 'days_since_update', 'visible_queries', 'rare_share', 'content_age_missing', 'update_age_missing', 'type_comparison article', 'type_feedly article', 'type_keyword article']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.